# 💬 Sentiment Analysis with LSTM
**Natural Language Processing with Keras/TensorFlow**
---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow version : {tf.__version__}')
print('Libraries loaded ✅')

## 2. Load & Explore Dataset
> Using a **Synthetic Product Reviews** dataset. The task is binary classification: predicting whether a review is Positive (1) or Negative (0) based on the text.

In [ ]:
# Generate dataset inline for reproducibility
np.random.seed(42)
positive_templates = [
    'I absolutely love this {noun}, it is {adj1} and {adj2}!',
    'This is the {adj1} {noun} I have ever experienced. Highly {verb}!',
    'Amazing quality, {adj1} performance, and {adj2} design.',
    'I am very {adj1} with this {noun}. It works {adv}.',
    'Fantastic {noun}! The features are {adj1} and the support is {adj2}.'
]
negative_templates = [
    'I absolutely hate this {noun}, it is {adj1} and {adj2}.',
    'This is the {adj1} {noun} I have ever bought. Do not {verb}!',
    'Terrible quality, {adj1} performance, and {adj2} design.',
    'I am very {adj1} with this {noun}. It works {adv}.',
    'Awful {noun}! The features are {adj1} and the support is {adj2}.'
]
nouns = ['product', 'movie', 'book', 'service', 'app', 'restaurant', 'hotel', 'device']
pos_adj = ['great', 'excellent', 'amazing', 'wonderful', 'fantastic', 'superb', 'brilliant']
neg_adj = ['terrible', 'awful', 'horrible', 'disappointing', 'poor', 'bad', 'worst']
pos_verbs = ['recommend', 'suggest', 'endorse']
neg_verbs = ['buy', 'try', 'waste money on']
pos_adv = ['perfectly', 'flawlessly', 'beautifully', 'smoothly']
neg_adv = ['poorly', 'badly', 'horribly', 'barely']

data = []
for _ in range(500):
    tmpl = np.random.choice(positive_templates)
    text = tmpl.format(noun=np.random.choice(nouns), adj1=np.random.choice(pos_adj), adj2=np.random.choice(pos_adj), verb=np.random.choice(pos_verbs), adv=np.random.choice(pos_adv))
    data.append({'text': text, 'sentiment': 1, 'label': 'Positive'})

for _ in range(500):
    tmpl = np.random.choice(negative_templates)
    text = tmpl.format(noun=np.random.choice(nouns), adj1=np.random.choice(neg_adj), adj2=np.random.choice(neg_adj), verb=np.random.choice(neg_verbs), adv=np.random.choice(neg_adv))
    data.append({'text': text, 'sentiment': 0, 'label': 'Negative'})

np.random.shuffle(data)
df = pd.DataFrame(data)

print(f'Shape   : {df.shape}')
print(f'Classes : {df["label"].value_counts().to_dict()}')
df.head()

## 3. Text Preprocessing Pipeline
> Neural networks require fixed-size numerical input. We must:
1. **Tokenize**: Convert words to integer IDs based on frequency.
2. **Pad/Truncate**: Ensure all sequences are the same length (`max_len`).
3. **Embed**: Map integers to dense vectors (handled by the Keras Embedding layer).

In [ ]:
VOCAB_SIZE = 2000
MAX_LEN = 20
EMBEDDING_DIM = 32

texts = df['text'].values
labels = df['sentiment'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

# Tokenize
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Sample word index: {list(tokenizer.word_index.items())[:5]}")

# Sequence conversion and padding
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding='post', truncating='post')

print(f'\\nX_train shape: {X_train_seq.shape}')
print(f'X_test shape : {X_test_seq.shape}')
print(f'Sample sequence: {X_train_seq[0]}')

## 4. Build the LSTM Model

In [ ]:
def build_sentiment_lstm(vocab_size, embed_dim, max_len, lstm_units=32, dropout=0.2):
    model = keras.Sequential(name='Sentiment_LSTM')
    
    # Embedding layer: learns dense representations of words
    model.add(layers.Embedding(vocab_size, embed_dim, input_length=max_len))
    
    # LSTM layer: processes the sequence of embeddings
    model.add(layers.LSTM(lstm_units, return_sequences=False))
    
    # Regularization
    model.add(layers.Dropout(dropout))
    
    # Optional dense layer for extra capacity
    model.add(layers.Dense(16, activation='relu'))
    model.add(layers.Dropout(dropout))
    
    # Output layer: sigmoid for binary probability
    model.add(layers.Dense(1, activation='sigmoid'))
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_sentiment_lstm(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, lstm_units=32, dropout=0.2)
model.summary()

## 5. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max', verbose=1)
]

history = model.fit(
    X_train_seq, y_train,
    validation_split=0.15,
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## 6. Training History

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist['loss'], color='#e05252', lw=2, label='Train')
axes[0].plot(hist['val_loss'], color='#06b6d4', lw=2, label='Val')
axes[0].set_title('Training Loss (Binary Crossentropy)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(hist['accuracy'], color='#34d399', lw=2, label='Train')
axes[1].plot(hist['val_accuracy'], color='#f59e0b', lw=2, label='Val')
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout(); plt.show()

## 7. Evaluate on Test Set

In [ ]:
y_pred_prob = model.predict(X_test_seq).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print('='*50)
print('          LSTM Sentiment Test Results')
print('='*50)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('='*50)
print('\\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative (0)', 'Positive (1)']))

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Negative (0)', 'Positive (1)'],
            yticklabels=['Negative (0)', 'Positive (1)'],
            linewidths=1, linecolor='white')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Live Prediction Function

In [ ]:
def predict_sentiment(text, model, tokenizer, max_len=20):
    # Preprocess
    seq = pad_sequences(tokenizer.texts_to_sequences([text]), maxlen=max_len, padding='post', truncating='post')
    # Predict
    prob = float(model.predict(seq, verbose=0).flatten()[0])
    label = "Positive 😊" if prob >= 0.5 else "Negative 😞"
    return label, prob

# Test with custom examples
test_reviews = [
    "I absolutely love this product, it is amazing and works perfectly!",
    "Terrible quality, awful performance, and horrible design.",
    "The movie was decent, though the pacing was a bit slow.",
    "This is the worst app I have ever bought. Do not try!"
]

print("--- Custom Predictions ---")
for review in test_reviews:
    label, prob = predict_sentiment(review, model, tokenizer, MAX_LEN)
    print(f"[{label:^18}] (p={prob:.4f}) | {review}")

## 10. Save Model & Tokenizer

In [ ]:
import os, joblib
os.makedirs('../models', exist_ok=True)
model.save('../models/sentiment_lstm.keras')

# Save tokenizer word index
joblib.dump(tokenizer.word_index, '../models/tokenizer_word_index.pkl')
print('Model saved  → models/sentiment_lstm.keras')
print('Tokenizer saved → models/tokenizer_word_index.pkl')

## 11. Key Takeaways
> - **Text must be numerical**: Tokenization and padding are mandatory preprocessing steps for NLP in neural networks.
> - **Embeddings are powerful**: They learn semantic relationships between words, vastly outperforming bag-of-words or TF-IDF for sequential models.
> - **LSTMs capture context**: By processing the sequence of word embeddings, the LSTM understands that "not good" is different from "good", unlike simpler models.
> - **OOV (Out of Vocabulary)**: Words not in the training vocabulary are mapped to the `<OOV>` token. A larger vocabulary size reduces OOV occurrences but increases model size.